In [ ]:
"""
Convert the QASPER rewrite parquet (raw Q&A fields) into Conversation-format 
parquet that LossEvalDataset can load for log perplexity evaluation.

Usage:
    python convert_rewrite_to_conversations.py <input_parquet> <output_parquet>

Example:
    python convert_rewrite_to_conversations.py \
        /home/vo43/cartridges/examples/arxiv/qasper_gpt41_rewrite.parquet \
        /home/vo43/cartridges/examples/arxiv/qasper_rewrite_eval.parquet
"""

import sys
import pandas as pd

from cartridges.structs import Conversation, write_conversations

# Same prompt template used in evals.py
PROMPT = """\
Please write a succinct answer to the following question.
You do not need to restate the paper name or answer in complete sentences.

<question>
{question}
</question>

Provide your answer in the following format (output nothing else):

<answer>
{{your answer here}}
</answer>"""


def convert(input_path: str, output_path: str):
    df = pd.read_parquet(input_path)
    print(f"Loaded {len(df)} rows from {input_path}")

    conversations = []
    for _, row in df.iterrows():
        convo = Conversation(
            messages=[
                Conversation.Message(
                    role="user",
                    content=PROMPT.format(question=row["question"]),
                    token_ids=None,
                ),
                Conversation.Message(
                    role="assistant",
                    content=f"<answer>\n{row['answer']}\n</answer>",
                    token_ids=None,
                ),
            ],
            system_prompt="",
            metadata={
                "paper_id": row["paper_id"],
                "title": row["title"],
            },
        )
        conversations.append(convo)

    write_conversations(conversations, output_path)
    print(f"Wrote {len(conversations)} conversations to {output_path}")


if __name__ == "__main__":
    if len(sys.argv) != 3:
        print(f"Usage: {sys.argv[0]} <input_parquet> <output_parquet>")
        sys.exit(1)
    convert(sys.argv[1], sys.argv[2])


In [4]:
import torch

cartridges = torch.load("/home/vo43/cartridges/outputs/2026-02-11-14-21-48-arxiv_train/7411f3e7-396c-4cb4-9ab6-58b19b983032/cache-step408.pt", map_location=torch.device("cpu"), weights_only=False)

In [67]:
import pandas as pd
path = "/home/vo43/cartridges/outputs/0_n128/7a282c8d-0eb6-4fa2-893a-6427f8e3d987/artifact/dataset.parquet"
df = pd.read_parquet(path).to_dict(orient="records")


In [78]:
evalqasper2 = pd.read_parquet("//home/vo43/cartridges/outputs/0_n128/7a282c8d-0eb6-4fa2-893a-6427f8e3d987/artifact/dataset.parquet").to_dict(orient="records")

In [82]:
evalqasper2[0]['messages']

array([{'content': 'What are the most effective features for distinguishing between positive and negative evidence in a question-answering system, given a question and a set of retrieved web pages as potential evidence?', 'role': 'user', 'token_ids': array([  3923,    527,    279,   1455,   7524,   4519,    369,  86055,
                1990,   6928,    323,   8389,   6029,    304,    264,   3488,
                  12,    598,     86,   4776,   1887,     11,   2728,    264,
                3488,    323,    264,    743,    315,  31503,   3566,   6959,
                 439,   4754,   6029,     30, 128009]), 'top_logprobs': None}                                                                                                                                                                                                                                                    ,
       {'content': "Based on the paper, the most effective features for distinguishing between positive and negative evid

In [107]:
evalqasper2[0].keys()

dict_keys(['messages', 'system_prompt', 'metadata', 'type'])

In [ ]:
evalqasper2[0][""]

'todo'

In [105]:
evalqasper[0].keys()

dict_keys(['paper_id', 'title', 'abstract', 'question', 'answer', 'old_answer', 'old_question'])

In [74]:
for row in evalqasper:
    for i in row["messages"]:
        print(i)

KeyError: 'messages'

In [69]:
for row in df:
    for i in row["messages"]:
        print(i)

{'content': 'What are the most effective features for distinguishing between positive and negative evidence in a question-answering system, given a question and a set of retrieved web pages as potential evidence?', 'role': 'user', 'token_ids': array([  3923,    527,    279,   1455,   7524,   4519,    369,  86055,
         1990,   6928,    323,   8389,   6029,    304,    264,   3488,
           12,    598,     86,   4776,   1887,     11,   2728,    264,
         3488,    323,    264,    743,    315,  31503,   3566,   6959,
          439,   4754,   6029,     30, 128009]), 'top_logprobs': None}
{'content': "Based on the paper, the most effective features for distinguishing between positive and negative evidence are:\n\n1. **q-e.comm (question-evidence common word feature)**: This feature indicates whether a word in the evidence also occurs in the question. A feature value of 1 suggests that the word may not be part of the answer, which can help distinguish positive from negative evidence.

In [94]:
import pandas as pd

df = pd.read_parquet("/home/vo43/cartridges/examples/arxiv/qasper_rewrite_eval.parquet").to_dict(orient="records")

In [98]:
df[0]["messages"][0]

{'content': 'Please write a succinct answer to the following question.\nYou do not need to restate the paper name or answer in complete sentences.\n\n<question>\nIn the paper "Question Answering based Clinical Text Structuring Using Pre-trained Language Model," what dataset is used to pretrain the language model?\n</question>\n\nProvide your answer in the following format (output nothing else):\n\n<answer>\n{your answer here}\n</answer>',
 'role': 'user',
 'token_ids': None,
 'top_logprobs': None}

In [93]:
df[0]["old_answer"]

{'evidence': array(['To implement deep neural network models, we utilize the Keras library BIBREF36 with TensorFlow BIBREF37 backend. Each model is run on a single NVIDIA GeForce GTX 1080 Ti GPU. The models are trained by Adam optimization algorithm BIBREF38 whose parameters are the same as the default settings except for learning rate set to $5\\times 10^{-5}$. Batch size is set to 3 or 4 due to the lack of graphical memory. We select BERT-base as the pre-trained language model in this paper. Due to the high cost of pre-training BERT language model, we directly adopt parameters pre-trained by Google in Chinese general corpus. The named entity recognition is applied on both pathology report texts and query texts.'],
       dtype=object),
 'extractive_spans': array(['Chinese general corpus'], dtype=object),
 'free_form_answer': '',
 'highlighted_evidence': array(['Due to the high cost of pre-training BERT language model, we directly adopt parameters pre-trained by Google in Chinese gene

In [57]:
from huggingface_hub import login, upload_file

login()  # will prompt for token (or set HF_TOKEN env var)

upload_file(
    path_or_fileobj="/scratch/scholar/vo43/qasper_65520.parquet",
    path_in_repo="qasper_65520.parquet",   # filename on the Hub
    repo_id="qtris123/qasper_self_study_65520",
    repo_type="dataset",
)

ImportError: The `notebook_login` function can only be used in a notebook (Jupyter or Colab) and you need the `ipywidgets` module: `pip install ipywidgets`.

In [1]:
"""Generate a QASPER context .txt file for KV cache initialization."""
from cartridges.data.qasper.resources import QASPERResource

OUTPUT_PATH = "qasper_context.txt"

resource = QASPERResource(QASPERResource.Config(topic="question"))
text = resource.to_string()

with open(OUTPUT_PATH, "w") as f:
    f.write(text)

print(f"Wrote {len(text)} chars to {OUTPUT_PATH}")


/home/vo43/.conda/envs/cartridges/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/vo43/.conda/envs/cartridges/lib/python3.12/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Wrote 465062 chars to qasper_context.txt


In [ ]:
import pandas as pd

df1 = pd.read_parquet("/scratch/scholar/vo43/qasper_30720.parquet")
df2 = pd.read_parquet("/home/vo43/cartridges/outputs/2026-02-19-01-42-50-qasper_synthesize_with_server/4b43ef43-686f-4047-83c8-8d2c80f5dca6/artifact/dataset.parquet")
 
combined = pd.concat([df1, df2], ignore_index=True)

combined.to_parquet("/scratch/scholar/vo43/qasper_40960.parquet", index=False)

In [3]:
import pandas as pd 
t2 = pd.read_parquet("/home/vo43/cartridges/outputs/2026-02-19-01-42-50-qasper_synthesize_with_server/4b43ef43-686f-4047-83c8-8d2c80f5dca6/artifact/dataset.parquet")

In [12]:
t2["messages"].keys()

RangeIndex(start=0, stop=24560, step=1)

In [ ]:
import pandas as pd

df1 = pd.read_parquet("/scratch/scholar/vo43/qasper_40960.parquet")
df2 = pd.read_parquet("/home/vo43/cartridges/outputs/3_24560/4b43ef43-686f-4047-83c8-8d2c80f5dca6/artifact/dataset.parquet")
 
combined = pd.concat([df1, df2], ignore_index=True)

combined.to_parquet("/scratch/scholar/vo43/qasper_65520.parquet", index=False)

In [9]:
import pandas as pd
import glob
import os

input_dir = "/home/vo43/cartridges/outputs/2_30720/960fc36e-5672-4385-977a-4049c2029363/checkpoints"
output_path = "/scratch/scholar/vo43/qasper_30720.parquet"

# get all parquet files
files = glob.glob(os.path.join(input_dir, "*.parquet"))


# read + concatenate
df = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)

# write single parquet
df.to_parquet(output_path, index=False)

print("Done.")


Done.


In [8]:
import pandas as pd
import glob
import os

input_dir = "/home/vo43/cartridges/outputs/0-1024/21c09b68-5254-44cb-8237-d1e5a643b02a/checkpoints"
output_path = "/scratch/scholar/vo43/qasper_1024.parquet"

# get all parquet files
files = glob.glob(os.path.join(input_dir, "*.parquet"))


# read + concatenate
df = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)

# write single parquet
df.to_parquet(output_path, index=False)

print("Done.")


Done.


In [5]:
qwen.iloc[0]["messages"]

array([{'content': 'What are the most effective features for distinguishing between positive and negative evidence in a question-answering system, given a question and a set of retrieved web pages as potential evidence?', 'role': 'user', 'token_ids': array([  3923,    527,    279,   1455,   7524,   4519,    369,  86055,
                1990,   6928,    323,   8389,   6029,    304,    264,   3488,
                  12,    598,     86,   4776,   1887,     11,   2728,    264,
                3488,    323,    264,    743,    315,  31503,   3566,   6959,
                 439,   4754,   6029,     30, 128009]), 'top_logprobs': None}                                                                                                                                                                                                                                                    ,
       {'content': "Based on the paper, the most effective features for distinguishing between positive and negative evid

In [45]:
import pandas as pd 

data = pd.read_parquet("/home/vo43/cartridges/outputs/2026-02-17-09-53-15-qasper_synthesize_with_server/21a1632f-ed21-4905-87a8-2dc7f1673730/artifact/dataset.parquet")

In [ ]:
data.iloc[12]["messages"]

array([{'content': "'Can you structure the information in Results of Exploring Question Understanding and Adaptation in Neural-Network-Based Question Answering in the following format: INI? Be sure to include precise information like any dates, times, names, and numerical values.'", 'role': 'user', 'token_ids': array([     6,   6854,    499,   6070,    279,   2038,    304,  18591,
                 315,  18491,   5620,  16225,  46551,    323,  59531,    367,
                 304,  61577,  11500,   2404,  61439,  16225,  22559,    287,
                 304,    279,   2768,   3645,     25,   2006,     40,     30,
                2893,   2771,    311,   2997,  24473,   2038,   1093,    904,
               13003,     11,   3115,     11,   5144,     11,    323,  35876,
                2819,   3238, 128009]), 'top_logprobs': None}                                                                                                                                                                     

In [35]:
data.iloc[65]["messages"]

array([{'content': '###   201200_#S pose_1m\n\n\n, and …. the function of the the �尽职l 0n the 0n x in the model. So, the function in the “sky" 0 in the 0-e-guess 0-#xxgbr 0% s 0, 0, the function. 0n ··· No, 0LayerDAS: 0, the function.00.05.g 0n. the 0s 0. the model.000.0 of the. i.e. the function. 0,0 (to 00n. the function. 0,0 (since the 0n0.S. 0 T, and.006 translateOn) 0n,0:  the-0n. 0-s since the function. the number of the 0n,0 (s �. 0n\u200b-\u200b\u200bin 0-$$0n$$$\n.0 $$, a, and the function, 0, the number of the function. I will now find the answer to the 0$$, 0n 0. the 0\u200b.0m x a, i: 0\u200b.n(0, 0.0.0.S.0. the function value of the 0-01. 0, 0. 0. 0, 00sp .n ， 0, 0. The 0n. the same as 0.0. 0, 01, 01mzg01, 01.0n. 006 assim. and 0. 01. 0n. 01m， 0n. and the lawyer. 0. 0, 0n, 01 0. 0. 01. the function. the 01, 01. 0n. 001 x-in. the 01. 01. 01. 0, 01, 01, 01, 01. 01. the function of the and 0, 01, the world. 01. 0.0. 0 0. ands: 01202', 'role': 'user', 'token_ids': array([14374

In [25]:
data.iloc[100]["messages"][0]['content']

'As a candidate to use in the跄 the no one\'s\n\nHere a job to print a different and "M" to the same as the different, - the)  -1\n\n1\n\n@ \n\n=2\n\n@2\n\n@ (one) - -1\n\n@ \n\n\n\n\n\n\n1\n\n@1\n\n=2\n\n@2\n\n@2\n\n2\n\n@ \n\n@2\n\n=2\n\n@2\n\nPost the same\n\n1.2\n\n@2\n\n--(1\n\n@2\n\n@2\n\nR\n\n2\n\n@2\n\n1\n\n2\n\n2.1\n\n\n (2: \n\n\n\n1\n\n(2\n\n-2\n\n- (2\n\n@2: ( Resp.2\n\n(1)\n\n1\n\n2\n\n-1\n\n\n\n--1\n\n-2\n\n-1\n\n-2\n\n- -2\n\n\n\n- -2\n\n-2\n\n-@2\n\n\n--2\n\n-2\n\n-2\n\n- \n\n-2\n\n- \n\n-2\n\n- \n\n\n\n-2\n\n- \n\n\n\n--2\n\n- \n\n\n\n-2\n\n-2\n\n-2\n\n--- \n\n- \n\n\n- \n\n\n-2\n\n-2\n\n-1\n\n-2\n\n- \n\n\n\n-1\n\n- \n\n\n\n-2\n\n-2\n\n-2\n\n- \n\n\n (The\n\n-2\n\n-2\n\n-2\n\n-2\n\n-1\n\n-2\n\n\n\n-2\n\n1\n\n-1\n2\n\n--2\n\n-2\n\n-2:2:2\n\n--2\n\n-2\n\n-2:2\n\n-2:2:2\n\n-1\n\n-2\n\n-2\n\n-2:2\n\n-2:2:2.1\n\n-2:2: 2:2:2.2\n\n-2:  2:2:2:2:2:2.2\n\n-1:2:2\n\n-2.2:2:2:2:2.2\n\n-2:2:2:2:2:2:2.2.2:2.2.2:2:2:2.2.2:2:1:2\n\n-1:2:2.2:2:2.2:2:1:2:1.2:1:1.2:2:2.1.1:1?1:'

In [ ]:
from datasets import load_dataset, concatenate_datasets
#train data
# Login using e.g. `huggingface-cli login` to access this dataset
llama_0 = load_dataset("hazyresearch/m07d11_longhealth_synthesize_llama-3.2-3b_p10_n65536-0")
llama_1 = load_dataset("hazyresearch/m07d11_longhealth_synthesize_llama-3.2-3b_p10_n65536-1")
llama_2 = load_dataset("hazyresearch/m07d11_longhealth_synthesize_llama-3.2-3b_p10_n65536-2") 

llama_0 = llama_0["train"]
llama_1 = llama_1["train"]
llama_2 = llama_2["train"]

llama_df = concatenate_datasets([llama_0, llama_1, llama_2])
llama_df.to_parquet("/scratch/scholar/vo43/llama_longhealth.parquet")

In [ ]:
# Longhealth train split is stored in: /scratch/scholar/vo43/train.parquet and /home/vo43/cartridges/train.parquet
for split, dset in ds.items():
    dset.to_parquet(f"/scratch/scholar/vo43/{split}.parquet") 


In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds_eval = load_dataset("hazyresearch/m07d11_longhealth_synthesize_qwen3-4b_p10_n65536-0")
for split, dset in ds_eval.items():
    dset.to_parquet(f"/scratch/scholar/vo43/{split}_eval.parquet") 

In [ ]:
# Step 1: Create a script to generate the text file
from cartridges.data.longhealth.resources import LongHealthResource

resource = LongHealthResource(LongHealthResource.Config(
    patient_ids=["patient_01", "patient_02", "patient_03", "patient_04", "patient_05", "patient_06", "patient_07", "patient_08", "patient_09", "patient_10",
    "patient_11", "patient_12", "patient_13", "patient_14"]  # Choose your patients
))
context_text = resource.to_string()

# Save to file
with open("examples/arxiv/longhealth_context.txt", "w") as f:
    f.write(context_text)

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("hazyresearch/m07d28_mtob_synthesize_qwen3-4b_n65536-0")

In [ ]:
# from datasets import load_dataset, concatenate_datasets
# import pandas as pd

# # Login using e.g. `huggingface-cli login` to access this dataset
# llama_0 = load_dataset("hazyresearch/m07d28_mtob_synthesize_llama-3.2-3b_n65536-0")
# llama_1 = load_dataset("hazyresearch/m07d28_mtob_synthesize_llama-3.2-3b_n65536-1")

# llama_0_df = next(iter(llama_0.values()))
# llama_1_df = next(iter(llama_1.values()))

# llama_0_df.to_parquet(f"/scratch/scholar/vo43/llama_0_mtob.parquet")

# llamas_df = concatenate_datasets([llama_0_df, llama_1_df])
# llamas_df.to_parquet(f"/scratch/scholar/vo43/llama_mtob.parquet")

In [ ]:
# from datasets import load_dataset, concatenate_datasets
# import pandas as pd

# qwen_0 = load_dataset("hazyresearch/m07d28_mtob_synthesize_qwen3-4b_n65536-0")
# qwen_1 = load_dataset("hazyresearch/m07d28_mtob_synthesize_qwen3-4b_n65536-1")

# qwen_0_df = next(iter(qwen_0.values()))
# qwen_1_df = next(iter(qwen_1.values()))

# qwen_0_df.to_parquet(f"/scratch/scholar/vo43/qwen_0_mtob.parquet")

# qwens_df = concatenate_datasets([qwen_0_df, qwen_1_df]) 
# qwens_df.to_parquet(f"/scratch/scholar/vo43/qwen_mtob.parquet")

In [1]:
import pandas as pd
with open("/scratch/scholar/vo43/llama_0_mtob.parquet", "rb") as f:
    llama_0 = pd.read_parquet(f)

In [2]:
len(llama_0)

65536

In [137]:
import pandas as pd

path = "/home/vo43/cartridges/longhealth_llama0__8196.xlsx"

xl = pd.ExcelFile(path, engine="calamine")
print(xl.sheet_names)

df = pd.read_excel(path, sheet_name="step 256", engine="calamine")

['step 0', 'step 128', 'step 256', 'step384', 'step438']


In [133]:
df.head()

,index,optimizer_step,prompt,answer,pred,convo_id,sample_idx,num_system_and_user_tokens,num_assistant_tokens,score,idx,extracted_pred
0,0,384,Please answer the question below about the fol...,4030 g,{Answer}\n3500 g,patient_15_9,0,140,16,False,0,NaN
1,2,384,Please answer the question below about the fol...,Increased from 10 mg to 25 mg.,{Answer}\nDecreased from 25 mg to 10 mg.,patient_19_15,0,201,16,True,2,NaN
2,4,384,Please answer the question below about the fol...,Intraventricular conduction disorder,{Answer}\nIntraventricular conduction disorder,patient_19_12,0,179,16,False,4,NaN
3,6,384,Please answer the question below about the fol...,Remained the same at 2 mg,{Answer}\nDecreased from 3 mg to 2 mg,patient_19_13,0,196,16,False,6,NaN
4,8,384,Please answer the question below about the fol...,It was not included in the medication regimen ...,{Answer}\nIt was not included in the medicatio...,patient_19_16,0,195,16,False,8,NaN


In [ ]:
def check(i, ans, pred):
    from difflib import SequenceMatcher
    def find_best_match(reference, candidates):
        return max(candidates, key=lambda x: SequenceMatcher(None, reference, x).ratio())

    # Extract the answer between <answer> and </answer> tags
    import re

    pred = pred.strip()
    patterns = [
        # Normal XML
        (r'<answer>\s*(.*?)\s*</answer>', re.IGNORECASE | re.DOTALL),

        # Hybrid: {answer} ... </answer>
        (r'\{[A-Za-z_][A-Za-z_ ]*\}\s*(.*?)\s*</answer>', re.IGNORECASE | re.DOTALL),

        # Curly tag only: {your answer} ...
        (r'\{[A-Za-z_][A-Za-z_ ]*\}\s*(.*)', re.IGNORECASE | re.DOTALL),

        # "The answer is:" ...
        (r'The answer is:\s*(.*)', re.IGNORECASE | re.DOTALL),
    ]
    
    for pattern, flags in patterns:
        pred_match = re.search(pattern, pred, flags)
        if pred_match:
            break

    if pred_match:
        extracted_pred = pred_match.group(1).strip().lower()
        extracted_pred = extracted_pred.strip().strip("{}")
        print(f"True {i}: answer is", ans , " | pred:", extracted_pred , " | ", pred)
        if extracted_pred == ans.lower().strip():
            print("Correct")
            print("------------------------------------------")
            return True
        print("------------------------------------------")
    else:
        print(f"False {i}: answer is", ans , " | pred:", pred)
        print("------------------------------------------")
    return False

In [85]:
text = "The answer is:\nHemoglobin levels"
pred_match = re.search(r'The answer is:\s*(.+)', text, re.IGNORECASE)
print(pred_match.group(1))

Hemoglobin levels


In [119]:
text = "Chest X-ray < Abdominal Ultrasound < CT Scan of Chest/Abdomen/Pelvis < Abdominal Ultrasound  < Abdominal MRI\ns</answer>"
text.strip("</answer>")

'Chest X-ray < Abdominal Ultrasound < CT Scan of Chest/Abdomen/Pelvis < Abdominal Ultrasound  < Abdominal MRI\n'

In [138]:
count = 0
for i in range(df.shape[0]):
    row = df.iloc[i]
    if check(i, str(row["answer"]), str(row["pred"])):
        count += 1
print(count)

True 0: answer is 4030 g  | pred: 4000 g  |  {Answer}
4000 g
------------------------------------------
True 1: answer is Increased from 10 mg to 25 mg.  | pred: your_answer  |  {Answer}
{YOUR_ANSWER}
------------------------------------------
True 2: answer is Intraventricular conduction disorder  | pred: ventricular tachycardia  |  {Answer}
Ventricular tachycardia
------------------------------------------
True 3: answer is Remained the same at 2 mg  | pred: decreased from 3 mg to 2 mg  |  {Answer}
Decreased from 3 mg to 2 mg
------------------------------------------
True 4: answer is It was not included in the medication regimen in March 2008.  | pred: it was not included in the medication regimen in march 2008.  |  {Answer}
It was not included in the medication regimen in March 2008.
Correct
------------------------------------------
True 5: answer is Hemoglobin levels  | pred: hemoglobin levels  |  {Answer}
Hemoglobin levels
Correct
------------------------------------------
True

In [84]:
count = 0
for i in range(df.shape[0]):
    row = df.iloc[i]
    if check(i, str(row["answer"]), str(row["pred"])):
        count += 1
print(count)

At 5: answer is Hemoglobin levels  | pred: The answer is:
Hemoglobin levels
------------------------------------------
At 24: answer is Echocardiography < Chest X-ray < Echocardiography < Echocardiography  < Echocardiography <  Abdominal Ultrasound < Cardiac Angiography and Catheterization < Abdominal Ultrasound < Cardiac MRI  | pred: Echocardiography < Chest X-ray < Echocardiography < Cardiac Catheterization < Abdominal Ultrasound < Echocardiography < Cardiac MRI
------------------------------------------
At 28: answer is Chest X-ray < Abdominal Ultrasound < CT Scan of Chest/Abdomen/Pelvis < Abdominal Ultrasound  < Abdominal MRI  | pred: Chest X-ray < Abdominal Ultrasound < CT Scan of Chest/Abdomen/Pelvis < Abdominal Ultrasound  < Abdominal MRI
</answer>
------------------------------------------
At 39: answer is IgG  | pred: IgE
------------------------------------------
At 41: answer is Slower infusion time  | pred: The answer is: Oral steroids.
-------------------------------------